# Customer Churn Intelligence — Train / Validation / Test Split

## Objective

Create a **stratified 70/15/15** split on the cleaned, leakage-approved feature set. Features remain in raw cleaned form — no scaling, encoding, imputation, SMOTE, or modeling.

**Stage:** Step 8 — Data Splitting only.

### Set roles

**Training set** — used to fit models and learned preprocessing (encoders, scalers, etc.).

**Validation set** — used for model comparison, hyperparameter tuning, calibration, and threshold optimization.

**Test set** — must remain **untouched** until the complete model and decision policy are frozen. Never use it for tuning, feature selection, threshold selection, model selection, or calibration decisions (AGENTS.md rule 3).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import RANDOM_STATE, run_split_pipeline

## 1. Create Stratified Split

Two-stage split with `random_state=42`:
1. **70% train** vs 30% temporary holdout (stratified on `Churn`)
2. Holdout split **50/50** into validation and test (~15% each)

In [ ]:
split, manifest, validation = run_split_pipeline()

print(f"Random state: {RANDOM_STATE}")
print(f"Split manifest saved to: {manifest['output_path']}")

## 2. Partition Shapes

In [ ]:
shape_summary = pd.DataFrame(
    {
        "Partition": ["Train", "Validation", "Test"],
        "X shape (rows, features)": [
            split.X_train.shape,
            split.X_val.shape,
            split.X_test.shape,
        ],
        "y length": [len(split.y_train), len(split.y_val), len(split.y_test)],
        "customerID length": [len(split.id_train), len(split.id_val), len(split.id_test)],
    }
)
shape_summary

## 3. Churn Distribution by Split

In [ ]:
def split_churn_table(y: pd.Series, name: str) -> pd.Series:
    counts = y.value_counts()
    pct = (y.value_counts(normalize=True) * 100).round(2)
    return pd.Series(
        {
            "Partition": name,
            "Rows": len(y),
            "No": int(counts.get("No", 0)),
            "Yes": int(counts.get("Yes", 0)),
            "Yes %": pct.get("Yes", 0.0),
        }
    )


churn_table = pd.DataFrame(
    [
        split_churn_table(split.y_train, "Train"),
        split_churn_table(split.y_val, "Validation"),
        split_churn_table(split.y_test, "Test"),
    ]
)
churn_table

## 4. Overlap Checks

In [ ]:
overlap_summary = pd.DataFrame(
    {
        "Check": [
            "Row index overlap (train ∩ val ∩ test)",
            "customerID overlap (train ∩ val ∩ test)",
            "Total rows accounted for",
            "Max churn % spread across splits",
            "Validation passed",
        ],
        "Result": [
            validation["row_index_overlap"],
            validation["customer_id_overlap"],
            validation["partition_rows"],
            validation["max_churn_pct_spread"],
            validation["passed"],
        ],
    }
)
overlap_summary

## 5. Sample Feature Preview (raw cleaned form)

In [ ]:
print("X_train dtypes (no preprocessing applied):")
display(split.X_train.dtypes)
split.X_train.head(3)

## Split Summary

- **Train:** 4,930 rows (~70%) — fit models and preprocessing
- **Validation:** 1,056 rows (~15%) — tune and compare models
- **Test:** 1,057 rows (~15%) — final evaluation only (frozen until policy is set)
- **Churn rate:** ~26.5% Yes in all three splits (stratified)
- **Overlap:** 0 row-index overlap; 0 customerID overlap
- **Artifacts:** `data/processed/split_manifest.json` stores indices + metadata only. Full split CSVs are **not** saved — partitions are reproducible from `cleaned_churn.csv` + manifest + `random_state=42`.

**Next step (not performed here):** Leakage-safe preprocessing pipelines fitted on training data only.